# NBA Injury Risk — Data Cleaning
## Injury Dataset
### Load & Filter by Date

The injury dataset spans from 1951 to 2023. We restrict the data to October 2015 onward to align with our study window of the 2015-16 through 2022-23 NBA seasons. October is used as the cutoff rather than January to capture the full season from opening night.

In [1]:
import pandas as pd

In [2]:
dest = "../data/raw"
file = 'NBA Player Injury Stats(1951 - 2023).csv'

df_injury = pd.read_csv(f'{dest}/{file}')

# drop the junk index
df_injury = df_injury.drop(columns=["Unnamed: 0"])

In [3]:
# filter to your window
df_injury["Date"] = pd.to_datetime(df_injury["Date"])
# df_injury = df_injury[df_injury["Date"].dt.year >= 2015]
df_injury = df_injury[df_injury["Date"] >= "2015-10-01"]
df_injury = df_injury.reset_index(drop=True)

print(f"Shape: {df_injury.shape}")
print(f"Date range: {df_injury['Date'].min()} → {df_injury['Date'].max()}")
print(f"\nNotes value counts (top 10):\n{df_injury['Notes'].value_counts().head(10)}")
print(f"\nNull counts:\n{df_injury.isnull().sum()}")
print(f"\nSample:\n{df_injury.head(10)}")

Shape: (15449, 5)
Date range: 2015-10-27 00:00:00 → 2023-04-16 00:00:00

Notes value counts (top 10):
Notes
activated from IL                                    7302
placed on IL                                         1980
placed on IL with NBA health and safety protocols     522
placed on IL with illness                             428
placed on IL with sprained left ankle                 275
placed on IL with sprained right ankle                265
placed on IL with sore left knee                      178
placed on IL with sore right knee                     176
placed on IL for rest                                 135
placed on IL with concussion                          110
Name: count, dtype: int64

Null counts:
Date               0
Team               0
Acquired        8099
Relinquished    7351
Notes              0
dtype: int64

Sample:
        Date       Team Acquired                   Relinquished  \
0 2015-10-27      Bulls      NaN              Cristiano Felicio   
1 2015-10-2

### Filter Non-Workload Injury Events

We retain only IL placements that are plausibly caused by workload accumulation. Activated from IL rows are removed because they represent return-to-play events, not injuries. Non-physical events such as illness, COVID-19 protocols, rest decisions, suspensions, and administrative moves are excluded because they cannot be predicted from game log features. Contact injuries (contusions, fractures, concussions, lacerations) are also excluded — Cohan et al. (2021) included these but found them the least predictable category, mainly because they are collision-driven rather than load-driven.

In [4]:
# =============================================================================
# INJURY DATA FILTERING
# =============================================================================
# We filter the injury dataset to retain only IL placements that are
# plausibly driven by workload accumulation. The rationale for each
# decision is documented below.
#
# REMOVED:
#   - "activated from IL": return-to-play events, not injury occurrences
#   - "placed on IL for rest": load management decisions by the team,
#     not actual injuries. Including these would introduce false positives
#     since a healthy player being rested looks identical to an injured one
#     in the notes.
#   - "placed on IL with NBA health and safety protocols": COVID-19 related,
#     not driven by physical workload in any way.
#   - "placed on IL with illness": illness is not workload-driven and
#     cannot be predicted from game log features.
#   - Contact injuries (keywords: "contusion", "bruised", "lacerated",
#     "fracture", "concussion"): caused by player-to-player collisions, not cumulative
#     fatigue. Cohan et al. (2021) included contact injuries but noted
#     METIC performed worst on this category precisely because contact
#     injuries are essentially random events. Since our project is
#     explicitly workload-based, including them would add noise and hurt
#     what we're trying to predict.
#
# KEPT:
#   - Muscle injuries (strains, pulls, tears of hamstrings, calves, quads):
#     directly driven by fatigue and accumulated minutes — the most
#     workload-sensitive category.
#   - Ligament injuries (ACL, ankle sprains, knee sprains): among the most
#     common NBA injuries and strongly associated with accumulated load
#     and inadequate rest between games.
#   - Back injuries: chronic in nature and accumulate over a season,
#     particularly in high-minute players. Highly relevant to rolling
#     workload features.
#   - Foot injuries: the third most common injury site in the NBA and
#     often caused by stress from repeated high-impact activity.
#   - Core injuries (hip flexors, groin, abdominal strains): driven by
#     explosive cutting and directional changes that accumulate over
#     a heavy schedule.
#   - Upper body injuries (shoulders, elbows, wrists): can be overuse-
#     driven (e.g. rotator cuff tendinitis from shooting volume) and
#     are worth retaining, though they are less workload-sensitive than
#     lower body injuries.
#   - Knee injuries (sore knee, knee inflammation, etc.): retained
#     separately since knee notes don't always use ligament/muscle
#     language but are clearly load-related.
#   - Sore/tightness notes (e.g. "sore left ankle", "tightness in lower
#     back"): these are early warning signs of overuse and are exactly
#     what a workload model should be able to predict.
# =============================================================================

remove_keywords = [
    "activated from IL",
    "placed on IL for rest",
    "health and safety",
    "illness",
    "contusion",
    "bruised",
    "lacerated",
    "fracture",
    "concussion",
    "head cold",
    "headache",
    "sinus",
    "sore throat",
    "strep throat",
    "throat",
    "tooth",
    "dental",
    "respiratory",
    "dehydration",
    "nausea",
    "gastro",
    "conjunctivitis",
    "pink eye",
    "neuropathy",
    "COVID",
    "coronavirus",
    "quarantine",
    "not with team",
    "personal reasons",
    "suspension",
    "flu",
    "stomach virus",
    "upset stomach",
    "appendectomy",
    "pneumonia",
    "mononucleosis",
    "root canal",
    "oral surgery",
    "eye procedure",
    "kidney stone",
    "arrhythmia",
    "vestibular",
    "medical tests",
    "ineligible",
    "CBC",
    "conditioning"
]

### Apply Filters

We apply the keyword filters to the Notes column and keep only rows where Relinquished is filled, confirming the event is an IL placement rather than an activation.

In [5]:
keep_mask = ~df_injury["Notes"].str.contains(
    "|".join(remove_keywords), case=False, na=False
)

# retain only IL placements (Relinquished populated)
placement_mask = df_injury["Relinquished"].notna()

df_injury_filtered = df_injury[keep_mask & placement_mask].reset_index(drop=True)

df_injury_filtered = df_injury_filtered[
    df_injury_filtered["Notes"] != "placed on IL (P)"
].reset_index(drop=True)

print(f"Original shape:  {df_injury.shape}")
print(f"Filtered shape:  {df_injury_filtered.shape}") # 6264
print(f"Removed:         {len(df_injury) - len(df_injury_filtered)} rows")
print(f"\nRemaining Notes (top 15):\n{df_injury_filtered['Notes'].value_counts().head(15)}")

Original shape:  (15449, 5)
Filtered shape:  (6170, 5)
Removed:         9279 rows

Remaining Notes (top 15):
Notes
placed on IL                                  1980
placed on IL with sprained left ankle          275
placed on IL with sprained right ankle         265
placed on IL with sore left knee               178
placed on IL with sore right knee              176
placed on IL with left knee injury             102
placed on IL with sore lower back               91
placed on IL with sore left ankle               83
placed on IL with right knee injury             78
placed on IL with sore right ankle              71
placed on IL with left hamstring injury         59
placed on IL with strained left hamstring       55
placed on IL with back injury                   51
placed on IL with strained right hamstring      50
placed on IL with back spasms                   49
Name: count, dtype: int64


In [6]:
# print(df_injury_filtered["Notes"].value_counts().to_string())
df_injury_filtered.head(5)

,Date,Team,Acquired,Relinquished,Notes
0,2015-10-27,Bulls,NaN,Cristiano Felicio,placed on IL
1,2015-10-27,Bulls,NaN,Mike Dunleavy Jr.,placed on IL recovering from surgery on back
2,2015-10-27,Cavaliers,NaN,Iman Shumpert,placed on IL recovering from surgery on right ...
3,2015-10-27,Hawks,NaN,Tim Hardaway Jr.,placed on IL
4,2015-10-27,Hawks,NaN,Walter Tavares / Edy Tavares,placed on IL


### Checking Names to Merge with Player Game Logs.

Before matching the injury dataset to NBA API player IDs, we examine the Relinquished column for formatting issues. Pro Sports Transactions uses inconsistent naming conventions, including dual names separated by slashes, legal names in parentheses, and suffix variants that could prevent exact matching.

In [7]:
# print(df_injury_filtered["Relinquished"].value_counts().to_string())

In [8]:
dtd_mask = df_injury_filtered["Relinquished"].str.contains("DTD", na=False)
df_injury_filtered[dtd_mask]

,Date,Team,Acquired,Relinquished,Notes
3933,2020-12-27,Timberwolves,NaN,left wrist injury (DTD),placed on IL with left wrist injury (DTD)
5783,2023-01-06,Lakers,NaN,strained left quadriceps (DTD),placed on IL with strained left quadriceps
5949,2023-02-27,Magic,NaN,left knee injury (DTD),placed on IL with left knee injury


### Drop Malformed Rows

Three rows where the `Relinquished` column contained injury descriptions instead of player names have been removed. These errors resulted from a CSV parsing problem where a missing player name shifted all subsequent columns. Cross-referencing with Pro Sports Transactions confirmed that the player identities could not be confidently recovered.

In [9]:
df_injury_filtered = df_injury_filtered[~dtd_mask].reset_index(drop=True)

print(f"Filtered shape: {df_injury_filtered.shape}")

Filtered shape: (6167, 5)


In [10]:
# print(df_injury_filtered["Team"].value_counts().to_string())
# df_injury_filtered[df_injury_filtered["Relinquished"] == '76ers']
# 2018-12-27
# df_injury_filtered[df_injury_filtered["Date"] == "2018-12-27"]

### Map Team Names to Abbreviations

The injury dataset uses full team nicknames, while the NBA API uses two- or three-letter abbreviations. We create a mapping to standardize the Team column so it can be used for validation during the player ID matching step.

In [11]:
team_name_to_abbrev = {
    "76ers": "PHI",
    "Bucks": "MIL",
    "Bulls": "CHI",
    "Cavaliers": "CLE",
    "Celtics": "BOS",
    "Clippers": "LAC",
    "Grizzlies": "MEM",
    "Hawks": "ATL",
    "Heat": "MIA",
    "Hornets": "CHA",
    "Jazz": "UTA",
    "Kings": "SAC",
    "Knicks": "NYK",
    "Lakers": "LAL",
    "Magic": "ORL",
    "Mavericks": "DAL",
    "Nets": "BKN",
    "Nuggets": "DEN",
    "Pacers": "IND",
    "Pelicans": "NOP",
    "Pistons": "DET",
    "Blazers": "POR",
    "Raptors": "TOR",
    "Rockets": "HOU",
    "Spurs": "SAS",
    "Suns": "PHX",
    "Thunder": "OKC",
    "Timberwolves": "MIN",
    "Warriors": "GSW",
    "Wizards": "WAS",
}

df_injury_filtered["TEAM_ABBREVIATION"] = df_injury_filtered["Team"].map(team_name_to_abbrev)

In [12]:
# verify nothing was missed
print(df_injury_filtered["TEAM_ABBREVIATION"].isna().sum())

0


In [13]:
df_injury_filtered.head()

,Date,Team,Acquired,Relinquished,Notes,TEAM_ABBREVIATION
0,2015-10-27,Bulls,NaN,Cristiano Felicio,placed on IL,CHI
1,2015-10-27,Bulls,NaN,Mike Dunleavy Jr.,placed on IL recovering from surgery on back,CHI
2,2015-10-27,Cavaliers,NaN,Iman Shumpert,placed on IL recovering from surgery on right ...,CLE
3,2015-10-27,Hawks,NaN,Tim Hardaway Jr.,placed on IL,ATL
4,2015-10-27,Hawks,NaN,Walter Tavares / Edy Tavares,placed on IL,ATL


In [14]:
df_injury_filtered[df_injury_filtered["Relinquished"] == ' 76ers']
#df_injury_filtered[df_injury_filtered["Date"] == '2018-12-27']

,Date,Team,Acquired,Relinquished,Notes,TEAM_ABBREVIATION
2798,2018-12-27,76ers,NaN,76ers,placed on IL,PHI


### Inspect Player Name Issues

We identify two categories of naming inconsistencies: dual names where Pro Sports Transactions lists both a legal name and a common name separated by a slash, and parenthetical suffixes used to distinguish players with the same name. Both issues must be addressed before fuzzy matching.

In [15]:
# dual names
dual = df_injury_filtered["Relinquished"].str.contains("/", na=False)
print(f"Dual names: {dual.sum()}")
# print(df_injury_filtered[dual]["Relinquished"].value_counts().to_string())

Dual names: 323


In [16]:
# parentheticals
paren = df_injury_filtered["Relinquished"].str.contains(r"\(", na=False)
print(f"\nParentheticals: {paren.sum()}")
print(df_injury_filtered[paren]["Relinquished"].value_counts().to_string())


Parentheticals: 122
Relinquished
James Michael McAdoo / James McAdoo (Michael)    25
John Wall (Hildred)                              19
(William) Tony Parker                            16
John Collins (Martin)                            10
(James) Mike Scott                                9
Ed Davis (Adam)                                   7
Reggie Jackson (Shon)                             7
Justin Jackson (Aaron)                            7
Roy Devyn Marble / Roy Marble (Devyn)             4
Herbert Jones / Herb Jones (Keyshawn)             4
David Johnson (Ricardo)                           4
Gerald Green (b. 1986-01-26)                      3
Larry Sanders (b. 1988-11-21)                     3
Norris Cole (Gene)                                2
Justin Hamilton (Anthony)                         2


### Clean Player Names

We create a cleaning function that extracts the last segment of slash-separated dual names and strips parenthetical suffixes. This results in a single, standardized name per player that can be matched against the NBA API's PLAYER_NAME field.

In [17]:
import re

def clean_player_name(name):
    if pd.isna(name):
        return name
    
    # take last segment if multiple slashes
    if "/" in name:
        name = name.split("/")[-1]
    
    # remove anything in parentheses
    name = re.sub(r"\(.*?\)", "", name)
    
    # strip extra whitespace
    name = name.strip()
    
    return name

In [18]:
#df_injury_filtered["player_name_clean"] = df_injury_filtered["Relinquished"].apply(clean_player_name)

# spot check
# print(df_injury_filtered[dual | paren][["Relinquished", "player_name_clean"]].drop_duplicates().to_string())

In [19]:
print(df_injury_filtered[dual | paren][["Relinquished"]].drop_duplicates().to_string())

                                                           Relinquished
4                                          Walter Tavares / Edy Tavares
5                                                    Norris Cole (Gene)
22                                 Amare Stoudemire / Amar'e Stoudemire
30                                        Louis Amundson / Lou Amundson
32                                       Metta World Peace / Ron Artest
33                                Roy Devyn Marble / Roy Marble (Devyn)
146                                    Emanuel Ginobili / Manu Ginobili
173                       James Michael McAdoo / James McAdoo (Michael)
184                           Jose Juan Barea / Jose Barea / J.J. Barea
188                              Nene / Nene Hilario / Maybyner Hilario
235                                                  (James) Mike Scott
265                                      Maurice Williams / Mo Williams
266                                         Alexis Ajinca / Alex

In [20]:
# df_injury_filtered[df_injury_filtered["Relinquished"].str.contains("Metta", na=False)]["Relinquished"].unique()

In [21]:
# df_injury_filtered[df_injury_filtered["Relinquished"].str.contains("Nene", na=False)]["Relinquished"].unique()

### Apply Manual Name Overrides

##### There's leading space. We will use strip() to remove leading and trailing whitespace.

Some dual names need manual correction before cleaning because the part after the slash is not the name used by the NBA API. For example, Metta World Peace / Ron Artest would incorrectly resolve to Ron Artest. We also remove leading and trailing whitespace from several entries.

In [22]:
manual_overrides = {
    # name corrections - applied to Relinquished before cleaning
    "Metta World Peace / Ron Artest": "Metta World Peace",
    "Nene / Nene Hilario / Maybyner Hilario": "Nene",
    "Milos Teodosic / Milos Tedosic": "Milos Teodosic",
    "Wesley Matthews / Wes Matthews Jr.": "Wesley Matthews",
    "Sheldon McClellan / Sheldon Mac": "Sheldon McClellan",
    "Zhou Qi / Qi Zhou": "Zhou Qi",
    "Nazareth Mitrou-Long / Naz Mitrou-Long / Naz Long": "Naz Mitrou-Long",
    "Moritz Wagner / Moe Wagner": "Moritz Wagner",
    "Maurice Harkless / Moe Harkless": "Maurice Harkless",
    "Jacob Wiley / Jake Wiley": "Jacob Wiley",
}

# remove leading and trailing whitespace
df_injury_filtered["Relinquished"] = df_injury_filtered["Relinquished"].str.strip()

df_injury_filtered["Relinquished"] = df_injury_filtered["Relinquished"].replace(manual_overrides)
df_injury_filtered["player_name_clean"] = df_injury_filtered["Relinquished"].apply(clean_player_name)

In [23]:
print(df_injury_filtered[dual | paren][["Relinquished", "player_name_clean"]].drop_duplicates().to_string())

                                                          Relinquished   player_name_clean
4                                         Walter Tavares / Edy Tavares         Edy Tavares
5                                                   Norris Cole (Gene)         Norris Cole
22                                Amare Stoudemire / Amar'e Stoudemire   Amar'e Stoudemire
30                                       Louis Amundson / Lou Amundson        Lou Amundson
32                                                   Metta World Peace   Metta World Peace
33                               Roy Devyn Marble / Roy Marble (Devyn)          Roy Marble
146                                   Emanuel Ginobili / Manu Ginobili       Manu Ginobili
173                      James Michael McAdoo / James McAdoo (Michael)        James McAdoo
184                          Jose Juan Barea / Jose Barea / J.J. Barea          J.J. Barea
188                                                               Nene                Nene

In [24]:
# print(df_injury_filtered["player_name_clean"].value_counts().to_string())
# df_injury_filtered[df_injury_filtered["player_name_clean"] == "Sheldon McClellan / Sheldon Mac"]
# Relinquished 

##### We got a weird player name '76ers'. Only one row. We will remove this player

In [25]:
df_injury_filtered[df_injury_filtered["Relinquished"] == '76ers']

,Date,Team,Acquired,Relinquished,Notes,TEAM_ABBREVIATION,player_name_clean
2798,2018-12-27,76ers,NaN,76ers,placed on IL,PHI,76ers


In [26]:
df_injury_filtered = df_injury_filtered[
    df_injury_filtered["player_name_clean"] != "76ers"
].reset_index(drop=True)

df_injury_filtered[df_injury_filtered["player_name_clean"] == '76ers']

,Date,Team,Acquired,Relinquished,Notes,TEAM_ABBREVIATION,player_name_clean


## Pre-Match Sanity Checks

Before running fuzzy matching we verify that no nulls or empty strings remain in player_name_clean, confirm the manual overrides resolved correctly, and check that the team abbreviation mapping has no gaps.

In [27]:
# Null check on player_name_clean
print(df_injury_filtered["player_name_clean"].isna().sum())

# Empty strings after cleaning
print((df_injury_filtered["player_name_clean"] == "").sum())

# Verify the manual overrides worked
check_names = ["Metta World Peace", "Nene", "Milos Teodosic", "Wesley Matthews", "Sheldon McClellan", "Zhou Qi", "Naz Mitrou-Long"]
for name in check_names:
    count = (df_injury_filtered["player_name_clean"] == name).sum()
    print(f"{name}: {count}")

# Verify team mapping has no nulls
print(df_injury_filtered["TEAM_ABBREVIATION"].isna().sum())

0
0
Metta World Peace: 18
Nene: 7
Milos Teodosic: 1
Wesley Matthews: 16
Sheldon McClellan: 7
Zhou Qi: 4
Naz Mitrou-Long: 1
0


## Fuzzy Match
### Build Player Name Lookup from Game Logs

We create a name-to-ID dictionary from the game logs, using PLAYER_NAME as the key and PLAYER_ID as the value. This dictionary functions as a lookup table for matching cleaned injury dataset names to NBA API player IDs.

In [28]:
from rapidfuzz import process, fuzz

df_game_logs = pd.read_csv("../data/processed/player_game_logs_all.csv")

name_to_id = (
    df_game_logs[["PLAYER_ID", "PLAYER_NAME"]]
    .drop_duplicates()
    .set_index("PLAYER_NAME")["PLAYER_ID"]
    .to_dict()
)

print(f"Unique players in game logs: {len(name_to_id)}")

Unique players in game logs: 1248


### Fuzzy Match Function

We use Rapidfuzz's token_sort_ratio scorer to match each cleaned player name with the lookup table. A threshold of 90 is set — matches below this score are flagged for manual review instead of automatically assigning an ID.

In [29]:
def fuzzy_match_player(name, lookup, threshold=90):
    if pd.isna(name):
        return None, None, None
    
    match, score, _ = process.extractOne(
        name, lookup.keys(), scorer=fuzz.token_sort_ratio
    )
    
    if score >= threshold:
        return match, lookup[match], score
    return match, None, score  # return match but no ID if below threshold

### Fuzzy Match Player Names to NBA API IDs

We use the fuzzy matching function on each row in the injury dataset and review all matches below the confidence threshold to decide if they are true mismatches or edge cases that can be fixed.

In [30]:
results = df_injury_filtered["player_name_clean"].apply(
    lambda x: pd.Series(fuzzy_match_player(x, name_to_id), index=["matched_name", "PLAYER_ID", "match_score"])
)

df_injury_filtered = pd.concat([df_injury_filtered, results], axis=1)

# inspect low confidence matches
low_confidence = df_injury_filtered[
    (df_injury_filtered["match_score"] < 90) | 
    (df_injury_filtered["PLAYER_ID"].isna())
]


In [31]:
print(f"Low confidence or unmatched: {len(low_confidence)}")
print(low_confidence[["player_name_clean", "matched_name", "match_score"]].drop_duplicates().to_string())

Low confidence or unmatched: 286
          player_name_clean          matched_name  match_score
1         Mike Dunleavy Jr.         Mike Dunleavy    86.666667
13          Tony Wroten Jr.           Tony Wroten    84.615385
15     Gerald Henderson Jr.      Gerald Henderson    88.888889
33               Roy Marble         Royce O'Neale    60.869565
50          Martell Webster         Briante Weber    64.285714
90           Nikola Vucevic        Nikola Vučević    85.714286
173            James McAdoo  James Michael McAdoo    75.000000
186        Ray McCallum Jr.          Ray McCallum    85.714286
525       Jonas Valanciunas     Jonas Valančiūnas    88.235294
588             Wes Johnson        Wesley Johnson    88.000000
610              C.J. Miles              CJ Miles    88.888889
646          Reggie Bullock    Reggie Bullock Jr.    87.500000
689      George Papagiannis  Georgios Papagiannis    89.473684
694   Stephen Zimmerman Jr.     Stephen Zimmerman    89.473684
790              J.R. 

In [32]:
# df_injury_filtered[df_injury_filtered['player_name_clean'] == 'Chris Wilcox']
# 1179           Chris Wilcox           C.J. Wilcox    69.565217
# Don't match
# df_injury_filtered[df_injury_filtered['player_name_clean'] == 'Eric Griffin']

### Post-Match Corrections

Names that scored below the threshold due to accented characters, Jr./Sr. suffixes, or nicknames are corrected using a second override dictionary, and the fuzzy match is re-run to ensure all resolvable names now exceed the threshold. Remaining unmatched players are fringe or retired players outside the study window and will drop naturally during the join with game logs.

In [33]:
post_clean_overrides = {
    "Nikola Vucevic": "Nikola Vučević",
    "Jonas Valanciunas": "Jonas Valančiūnas",
    "Kristaps Porzingis": "Kristaps Porziņģis",
    "Luka Doncic": "Luka Dončić",
    "Vit Krejci": "Vít Krejčí",
    "Anzejs Pasecniks": "Anžejs Pasečņiks",
    "Vlatko Cancar": "Vlatko Cancar",  # check exact NBA API spelling
    "C.J. Miles": "CJ Miles",
    "J.R. Smith": "JR Smith",
    "J.T. Thor": "JT Thor",
    "Herb Jones": "Herbert Jones",
    "Marcus Morris": "Marcus Morris Sr.",
    "Jimmy Butler": "Jimmy Butler III",
    "Frank Mason": "Frank Mason III",
    "Reggie Bullock": "Reggie Bullock Jr.",
    "Naz Mitrou-Long": "Naz Mitrou-Long",  # verify NBA API spelling
}

df_injury_filtered["player_name_clean"] = df_injury_filtered["player_name_clean"].replace(post_clean_overrides)

In [34]:
df_injury_filtered = df_injury_filtered.drop(columns=["matched_name", "PLAYER_ID", "match_score"])

In [35]:
results = df_injury_filtered["player_name_clean"].apply(
    lambda x: pd.Series(fuzzy_match_player(x, name_to_id), index=["matched_name", "PLAYER_ID", "match_score"])
)

df_injury_filtered = pd.concat([df_injury_filtered, results], axis=1)

# inspect low confidence matches
low_confidence = df_injury_filtered[
    (df_injury_filtered["match_score"] < 90) | 
    (df_injury_filtered["PLAYER_ID"].isna())
]

In [36]:
print(f"Low confidence or unmatched: {len(low_confidence)}")
print(low_confidence[["player_name_clean", "matched_name", "match_score"]].drop_duplicates().to_string())

Low confidence or unmatched: 132
          player_name_clean          matched_name  match_score
1         Mike Dunleavy Jr.         Mike Dunleavy    86.666667
13          Tony Wroten Jr.           Tony Wroten    84.615385
15     Gerald Henderson Jr.      Gerald Henderson    88.888889
33               Roy Marble         Royce O'Neale    60.869565
50          Martell Webster         Briante Weber    64.285714
173            James McAdoo  James Michael McAdoo    75.000000
186        Ray McCallum Jr.          Ray McCallum    85.714286
588             Wes Johnson        Wesley Johnson    88.000000
689      George Papagiannis  Georgios Papagiannis    89.473684
694   Stephen Zimmerman Jr.     Stephen Zimmerman    89.473684
888         Mike Conley Jr.           Mike Conley    84.615385
1179           Chris Wilcox           C.J. Wilcox    69.565217
1264      Sheldon McClellan           Sheldon Mac    71.428571
1496       Wayne Selden Jr.          Wayne Selden    85.714286
1525           Eric Gr

In [37]:
# df_injury_filtered[df_injury_filtered['player_name_clean'] == 'Vlatko Cancar']
post_clean_overrides = {
    "Mike Dunleavy Jr.": "Mike Dunleavy",
    "Tony Wroten Jr.": "Tony Wroten",
    "Gerald Henderson Jr.": "Gerald Henderson",
    "Ray McCallum Jr.": "Ray McCallum",
    "Wes Johnson": "Wesley Johnson",
    "George Papagiannis": "Georgios Papagiannis",
    "Stephen Zimmerman Jr.": "Stephen Zimmerman",
    "Mike Conley Jr.": "Mike Conley",
    "Wayne Selden Jr.": "Wayne Selden",
    "Nigel Hayes": "Nigel Hayes-Davis",
    "Matt Williams": "Matt Williams Jr.",
    "R.J. Nembhard Jr.": "Ruben Nembhard Jr.",
    "Sheldon McClellan": "Sheldon Mac",  # NBA API uses this name
    "James McAdoo": "James Michael McAdoo",  # verify NBA API spelling
    'Vlatko Cancar': 'Vlatko Čančar' # corrected spelling
}

df_injury_filtered["player_name_clean"] = df_injury_filtered["player_name_clean"].replace(post_clean_overrides)

In [38]:
df_injury_filtered = df_injury_filtered.drop(columns=["matched_name", "PLAYER_ID", "match_score"])

In [39]:
results = df_injury_filtered["player_name_clean"].apply(
    lambda x: pd.Series(fuzzy_match_player(x, name_to_id), index=["matched_name", "PLAYER_ID", "match_score"])
)

df_injury_filtered = pd.concat([df_injury_filtered, results], axis=1)

# inspect low confidence matches
low_confidence = df_injury_filtered[
    (df_injury_filtered["match_score"] < 90) | 
    (df_injury_filtered["PLAYER_ID"].isna())
]

In [40]:
print(f"Low confidence or unmatched: {len(low_confidence)}")
print(low_confidence[["player_name_clean", "matched_name", "match_score"]].drop_duplicates().to_string())

Low confidence or unmatched: 14
     player_name_clean   matched_name  match_score
33          Roy Marble  Royce O'Neale    60.869565
50     Martell Webster  Briante Weber    64.285714
1179      Chris Wilcox    C.J. Wilcox    69.565217
1525      Eric Griffin     AJ Griffin    72.727273
1540    Billy Ouattara  Billy Garrett    66.666667
1560        Mike Young     Nick Young    80.000000
4616   B.J. Boston Jr.     BJ Johnson    56.000000
4623    Daulton Hommes    Jalen Jones    56.000000


## Game Logs Cleaning
### Need to further clean Game Logs

We load the combined game logs and remove columns that are not relevant to injury risk prediction. League ranking columns are dropped because they reflect relative performance rather than workload. Fantasy points, double-double and triple-double flags, and administrative columns are also eliminated. Rows where a player logged zero minutes are dropped since no workload features can be derived from them.

In [41]:
# load game logs
df_game_logs = pd.read_csv("../data/processed/player_game_logs_all.csv")

# drop irrelevant columns
rank_cols = [col for col in df_game_logs.columns if col.endswith("_RANK")]

drop_cols = rank_cols + [
    "NICKNAME",
    "NBA_FANTASY_PTS",
    "WNBA_FANTASY_PTS",
    "DD2",
    "TD3",
    "AVAILABLE_FLAG",
    "MIN_SEC",
    "TEAM_COUNT",
]

df_game_logs = df_game_logs.drop(columns=drop_cols)

# drop zero minute rows
df_game_logs = df_game_logs[df_game_logs["MIN"] > 0].reset_index(drop=True)

# fix dtypes
df_game_logs["GAME_DATE"] = pd.to_datetime(df_game_logs["GAME_DATE"])
df_injury_filtered["Date"] = pd.to_datetime(df_injury_filtered["Date"])
df_injury_filtered["PLAYER_ID"] = df_injury_filtered["PLAYER_ID"].astype("Int64")
df_game_logs["PLAYER_ID"] = df_game_logs["PLAYER_ID"].astype("Int64")

In [42]:
print(f"Shape after cleaning: {df_game_logs.shape}")
print(f"Zero minute rows: {(df_game_logs['MIN'] == 0).sum()}")
print(f"Null MIN rows: {df_game_logs['MIN'].isna().sum()}")
print(f"Duplicate rows: {df_game_logs.duplicated().sum()}")

Shape after cleaning: (216088, 33)
Zero minute rows: 0
Null MIN rows: 0
Duplicate rows: 0


## Combine game logs with injury data

We assign a binary injury indicator to each game log entry using a 30-day lookback period. For each IL placement in the injury dataset, all games the player participated in within 30 days prior to the placement date are marked as positive injury cases. Overlapping windows from multiple IL placements within 30 days of each other are combined into a single continuous window to prevent duplicate labeling.

In [43]:
# ### Combine game logs with injury data

def label_injury_windows(df_logs, df_injuries):
    df_logs = df_logs.copy()
    df_logs["injury_within_30_days"] = 0

    for player_id, group in df_injuries.groupby("PLAYER_ID"):
        dates = sorted(group["Date"].tolist())
        
        # merge overlapping 30-day windows
        merged_windows = []
        window_start = dates[0] - pd.Timedelta(days=30)
        window_end = dates[0]
        
        for date in dates[1:]:
            new_start = date - pd.Timedelta(days=30)
            if new_start <= window_end:
                window_end = max(window_end, date)
            else:
                merged_windows.append((window_start, window_end))
                window_start = new_start
                window_end = date
        
        merged_windows.append((window_start, window_end))
        
        for start, end in merged_windows:
            mask = (
                (df_logs["PLAYER_ID"] == player_id) &
                (df_logs["GAME_DATE"] >= start) &
                (df_logs["GAME_DATE"] <= end)
            )
            df_logs.loc[mask, "injury_within_30_days"] = 1
    
    return df_logs

In [44]:
df_game_logs = label_injury_windows(df_game_logs, df_injury_filtered)

print(f"Total game log rows: {len(df_game_logs)}")
print(f"Injury labeled rows: {df_game_logs['injury_within_30_days'].sum()}")
print(f"Non-injury rows: {(df_game_logs['injury_within_30_days'] == 0).sum()}")
print(f"Injury rate: {df_game_logs['injury_within_30_days'].mean():.4f}")
print(df_game_logs.groupby("SEASON_YEAR")["injury_within_30_days"].mean())

Total game log rows: 216088
Injury labeled rows: 39072
Non-injury rows: 177016
Injury rate: 0.1808
SEASON_YEAR
2015-16    0.118007
2016-17    0.147987
2017-18    0.181870
2018-19    0.185721
2019-20    0.178791
2020-21    0.194237
2021-22    0.232893
2022-23    0.208545
Name: injury_within_30_days, dtype: float64


We use a flexible injury-labeling system, keeping vague IL placements and notes on soreness or tightness, as these are early signs of overuse directly related to workload-based prediction. Although this leads to a higher positive class rate of 18% compared to stricter criteria, it more effectively captures the early injury signals our rolling window features are intended to identify.

Potential drawbacks in our methodology: 

1. Label noise: A player placed on IL on January 15th may have actually gotten injured on January 12th, January 10th, or even earlier. Labeling all games in the 30-day window as "pre-injury" but some of those early games may have been played when the player was completely healthy. This introduces noise into your positive class labels.
2. Underreporting and reporting delays: As Cohan himself acknowledges, teams are incentivized to obscure injuries for competitive advantage. The IL placement date may not reflect when the injury actually occurred, making the 30-day window an approximation at best. A player could have been playing hurt for weeks before being officially placed on IL.
3. Arbitrary window size: 30 days is not derived from any clinical or biomechanical evidence — it's a practical choice. Some injuries like ACL tears are acute and have no meaningful 30-day buildup, while others like stress fractures or tendinitis genuinely accumulate over months. A single window size treats all injury types the same way.
4. Overlapping windows: If a player gets injured twice within 30 days, their windows overlap and some games get labeled twice. This creates ambiguity in the label for those rows.
5. Healthy games mislabeled: A player who plays 25 games in 30 days before an injury had most of those games played while healthy. Labeling all 25 as pre-injury when only the last few may have been truly high-risk dilutes the signal. 
 
We adopt the 30-day lookback window following Cohan et al. (2021) to ensure methodological consistency and maximize positive class samples due to the severe class imbalance in the dataset, while recognizing that this introduces label noise since the exact injury occurrence is unknown.

In [45]:
df_game_logs.head()

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,...,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,is_playoff,injury_within_30_days
0,2016-17,203999,Nikola Jokić,1610612743,DEN,Denver Nuggets,21601225,2017-04-12,DEN @ OKC,W,...,4,1,5,2,4,10,29,9,0,0
1,2016-17,201935,James Harden,1610612745,HOU,Houston Rockets,21601224,2017-04-12,HOU vs. MIN,W,...,4,4,1,0,3,4,27,19,0,0
2,2016-17,1626157,Karl-Anthony Towns,1610612750,MIN,Minnesota Timberwolves,21601224,2017-04-12,MIN @ HOU,L,...,0,1,0,1,2,1,28,2,0,0
3,2016-17,203932,Aaron Gordon,1610612753,ORL,Orlando Magic,21601217,2017-04-12,ORL vs. DET,W,...,0,1,2,0,2,4,32,17,0,0
4,2016-17,202331,Paul George,1610612754,IND,Indiana Pacers,21601226,2017-04-12,IND vs. ATL,W,...,1,1,0,0,3,3,32,15,0,0


## Combine game logs and injury data with player's birthdates


In [46]:
# =============================================================================
# ADD PLAYER AGE
# =============================================================================
# CommonPlayerInfo is the only NBA API endpoint with BIRTHDATE.
# We fetched birthdates for all 1,248 unique players in the dataset
# and saved to player_birthdates.csv to avoid re-hitting the API.
# Age is computed at the time of each game (not static) since players
# age across the 8-season study window.
# Per Lu et al. (2022), age was a statistically significant predictor
# of lower extremity muscle strain in NBA athletes.
# =============================================================================

df_birthdates = pd.read_csv("../data/raw/player_birthdates.csv")
df_birthdates["BIRTHDATE"] = pd.to_datetime(df_birthdates["BIRTHDATE"])

df_game_logs = df_game_logs.merge(
    df_birthdates[["PLAYER_ID", "BIRTHDATE"]],
    on="PLAYER_ID",
    how="left"
)

df_game_logs["age"] = (
    (df_game_logs["GAME_DATE"] - df_game_logs["BIRTHDATE"]).dt.days / 365.25
)

df_game_logs = df_game_logs.drop(columns=["BIRTHDATE"])

In [47]:
# Verify
print(f"Null age count: {df_game_logs['age'].isna().sum()}")
print(f"Age range: {df_game_logs['age'].min():.1f} – {df_game_logs['age'].max():.1f} years")
print(df_game_logs[["PLAYER_NAME", "GAME_DATE", "age"]].head(10))

Null age count: 0
Age range: 18.8 – 43.1 years
          PLAYER_NAME  GAME_DATE        age
0        Nikola Jokić 2017-04-12  22.143737
1        James Harden 2017-04-12  27.627652
2  Karl-Anthony Towns 2017-04-12  21.407255
3        Aaron Gordon 2017-04-12  21.571526
4         Paul George 2017-04-12  26.945927
5        Kevin Durant 2017-04-12  28.533881
6    Jimmy Butler III 2017-04-12  27.575633
7    Hassan Whiteside 2017-04-12  27.830253
8      DeAndre Jordan 2017-04-12  28.725530
9        Clint Capela 2017-04-12  22.902122


In [51]:
df_game_logs.head()

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,...,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,is_playoff,injury_within_30_days,age
0,2016-17,203999,Nikola Jokić,1610612743,DEN,Denver Nuggets,21601225,2017-04-12,DEN @ OKC,W,...,1,5,2,4,10,29,9,0,0,22.143737
1,2016-17,201935,James Harden,1610612745,HOU,Houston Rockets,21601224,2017-04-12,HOU vs. MIN,W,...,4,1,0,3,4,27,19,0,0,27.627652
2,2016-17,1626157,Karl-Anthony Towns,1610612750,MIN,Minnesota Timberwolves,21601224,2017-04-12,MIN @ HOU,L,...,1,0,1,2,1,28,2,0,0,21.407255
3,2016-17,203932,Aaron Gordon,1610612753,ORL,Orlando Magic,21601217,2017-04-12,ORL vs. DET,W,...,1,2,0,2,4,32,17,0,0,21.571526
4,2016-17,202331,Paul George,1610612754,IND,Indiana Pacers,21601226,2017-04-12,IND vs. ATL,W,...,1,0,0,3,3,32,15,0,0,26.945927


## Save cleaned data

The cleaned injury dataset and labeled game logs are stored in the processed data directory. The injury dataset includes the matched player IDs and fuzzy match metadata for traceability. The labeled game logs are used as input for feature engineering in the next notebook.

In [49]:
df_injury_filtered.to_csv("../data/processed/injury_data_cleaned.csv", index=False)
print("Saved: injury_data_cleaned.csv")

Saved: injury_data_cleaned.csv


In [50]:
df_game_logs.to_csv("../data/processed/game_logs_labeled.csv", index=False)
print("Saved: game_logs_labeled.csv")

Saved: game_logs_labeled.csv
